# Coding Assistant Evaluations on Amazon Bedrock AgentCore

In this notebook, you build an evaluation for two kinds of coding agents, run it on Amazon Bedrock AgentCore Runtime, and learn why each part is there. A **PR review agent** reads a change and reports its defects. A **coding agent** reads an issue report and writes the fix. The agent under test is Codex on Amazon Bedrock. Each trial runs in its own AgentCore session, a fresh microVM, and a firewall inside the session controls what the agent can reach.

| | |
|---|---|
| **Time** | About 75 to 90 minutes: the cells run for about 35 minutes (image builds 2 to 8 minutes, trials about 10 minutes, deleting the runtimes 7 to 9 minutes), and the rest is reading |
| **Cost** | About $3 to $5 in your AWS account, with a 30 % uncertainty. Most of it is the agent model and the image builds |
| **What it creates** | 3 IAM roles, 1 ECR repository (about 7 GB of images), 1 S3 bucket, 1 CodeBuild project, CloudWatch log groups, and one AgentCore runtime for each task. All names start with one prefix, and all resources have the tag `Project=evals-streamline-spike`. Section 11 deletes them |

Read the notebook in order. The cells that cost money say so.

### Procedure

1. Learn what an eval for a coding agent is, and the approach of this module.
2. Check your prerequisites and permissions. Meet the models and the agent.
3. Create the resources in your AWS account (bootstrap).
4. Learn the task format, and examine a PR review datapoint and a coding datapoint, and the parts that grade them.
5. Build and deploy the task images, and check the git seal.
6. Run one PR review trial and one coding trial. Read the results.
7. Run the controls: an empty agent and the merged fix.
8. See what the firewall allows and what it blocks.
9. Run many trials at the same time, and calculate the dataset metrics.
10. Read the results with the noise in mind.
11. **Clean up.** Delete everything that this notebook made.

> **Warning: this notebook creates resources in your AWS account.**
>
> | | |
> |---|---|
> | **What it creates** | 3 IAM roles, 1 ECR repository (about 7 GB of images), 1 S3 bucket, 1 CodeBuild project, 1 CloudWatch log group, and one AgentCore runtime for each task. All have a name that starts with the prefix in section 2 (`evals-nb-` by default) and the tag `Project=evals-streamline-spike`. |
> | **Cost** | About $3 to $5 for the whole notebook, with a 30 % uncertainty. Most of it is the agent model (Codex on Bedrock) and the image builds. While you wait, the images cost about $0.02 per day. A runtime that is not in use costs nothing. |
> | **When you are done** | **Run section 11.** It deletes everything and shows lists that must be empty. If you stop early, run section 11 anyway: it works from a fresh kernel. |
>
> Use a sandbox account, not a shared production account.

## 1. What You Will Build and Learn

### What is an eval for a coding agent?

An eval is a set of examples where you already know the correct answer. You run the agent on each example and compare its output to the answer. The result is a score that you can track over time. If you are new to evals, start with the [Foundational Evaluations](../../../Foundational%20Evaluations/) modules.

For a chatbot, an example is a prompt and an expected response. For a coding agent, an example can be much larger. It could contain a full repository, a task in that repository, and the correct result.

This module evaluates two types of coding agents:

- A **PR review agent** reads a change and reports the defects. Then it approves the change or requests changes.
- A **coding agent** reads an issue and writes a patch that fixes it.

### The problem

Two problems make coding evals more difficult than other evals.

**The examples should come from your own code.** Public benchmarks, for example SWE-bench and [Code Review Bench](https://github.com/withmartian/code-review-benchmark), use PRs from popular open-source projects. They test for general defects, for example race conditions and missing null checks. A high score shows that the agent can find general defects. It does not show that the agent knows the rules of your team.

Here is an example. A coding agent opens a PR that moves a plan ID from Stripe metadata into your database. The agent does not know that a different billing service reads that metadata at each renewal. The tests for each service pass. One month later, the renewals reset customers to their old plans. A generic review agent approves this PR, because the change looks like a good cleanup. A reviewer who knows your system blocks it. To measure this skill, you need examples from your own code.

**Each example needs a full, sealed environment.** A review agent cannot review a diff alone. It must read the code around the change, and it must sometimes build the code and run the tests. Thus each example needs the repository at the correct commit, with all dependencies installed. The environment must also hide the answer. The correct fix is in a later commit, and the review comments are on GitHub. Agents search for data when they can. If the agent finds the answer, the score measures search, not review.

### The approach: your git history is the dataset

You already have a labeled dataset. Each PR in your repository records the code before the change, the change, what the reviewer said, and the fix that the reviewer approved. Your team did the labeling work during normal code review.

| Item | Where git keeps it | Use in the eval |
|------|--------------------|-----------------|
| The code before the change | The parent commit | The start state for the agent |
| The change | The diff from the parent commit to the PR commit | The input for a PR review agent |
| The reviewer decision | The PR review comments and the verdict | The answer for a PR review agent |
| The correct fix | The merged patch and its tests | The answer for a coding agent |

Each example is a commit hash and a PR number. Thus you can make the same example again at any time.

For each PR, the approach has three steps:

1. **Roll back the repository to the parent commit of the change.** The agent sees the code as the reviewer saw it, before the fix. If you start after the fix, the defect is not there.
2. **Seal the environment.** Remove all paths to the answer. Then the score measures the skill of the agent, not its ability to search.
3. **Use the PR review or the merged patch as the oracle.** The oracle is the gold standard answer. The grader compares the agent output to the oracle.

```
parent commit ──► PR change ──► review ──────────► fix ──► merge
      ▲               │            │                         │
  roll back here    input      oracle for a            oracle for a
                               PR review agent         coding agent
```

### How to seal the environment

A seal has three layers. Each layer closes one path to the answer.

| Layer | Path to the answer | Control |
|-------|--------------------|---------|
| **Git** | `git log --all` shows the fix commit | Clone only the parent commit. Remove the remote, the remote refs, and the reflog |
| **Network** | `gh pr view` shows the review comments | Allow only the model endpoint of the agent. Do not allow GitHub or a package index |
| **Verifier** | The agent reads or changes the hidden tests or the oracle | Keep them where the agent cannot read them, and copy them in only after the agent has stopped |

The tests are **hidden** for the same reason. If the agent could read the tests that grade it, it could write code that passes those tests and nothing else, and the score would measure that, not the fix.

### How a trial runs on AgentCore

A **trial** is one agent on one task. Each trial runs in one AgentCore Runtime **session**: a fresh microVM (2 vCPU, 8 GB) that starts from the task image and is thrown away after the trial. Thus each trial starts from the same state, and trials cannot change each other. Many trials run at the same time, one microVM each.

```
your machine (this notebook)          your AWS account, us-east-1
----------------------------          ------------------------------------------------------------------
p2_v2.py bootstrap   ---- API ---->   ECR repository, S3 bucket, CodeBuild project, 3 scoped IAM roles
p2_v2.py build       ---- zip ---->   S3 -> CodeBuild -> ECR image (task build file + firewall layer)
p2_v2.py deploy      ---- API ---->   one AgentCore runtime for each task
p2_v2.py run         ---- API ---->   one session = one microVM for each trial:

                                      +--------------------------------------------------------------+
                                      | 1. AGENT PHASE   Codex runs as user `agent` in an empty      |
                                      |    network namespace. Its only way out: a relay on localhost |
                                      |    to an allowlist proxy that forwards to the model endpoint |
                                      |    of the agent. Credentials: the agent role (that model).   |
                                      | 2. SWITCH        agent processes are killed, the allowlist   |
                                      |    changes to the judge endpoint, the hidden tests are       |
                                      |    copied in                                                  |
                                      | 3. GRADER PHASE  the task's own tests/test.sh runs.          |
                                      |    Credentials: the grader role (the judge model only).      |
                                      +--------------------------------------------------------------+
                      <---- tar ----  reward.json, agent logs, the patch or the findings
```

`p2_v2.py` is the runner. This notebook calls it as a command, so each step that you run here is a command that you can also run in a terminal. [README.md](README.md) and [TESTER-GUIDE.md](TESTER-GUIDE.md) in this folder have the reference.

### What you will learn

- The task format of the module: which files the agent sees, and which files only the grader sees.
- How the PR review datapoints and the coding datapoints were made, and why each was selected.
- How each kind is graded: matched findings and coverage for a PR review; a reward stack of hidden tests, a validation agent, a rubric judge, a taste judge, and patch bloat for a fix.
- How to read a `reward.json`, and how to find the cause of a failure.
- Why you run controls (the oracle and an empty agent) before you trust an agent score.
- How the seal is made and checked in an AgentCore session.
- How many trials you need before a pass rate means something.

> **Note:** The runner works in `us-east-1` only, with one agent (Codex on Bedrock), and the four tasks of the module.

## 2. Prerequisites

You need Python 3.11 or later, `boto3` 1.44.7 or later (older versions have no AgentCore `platformVersion`), and AWS credentials for a sandbox account in your terminal (`aws sso login`, `AWS_PROFILE`, or access keys). The Region is `us-east-1`. You do not need a container engine, Node, or the AWS CLI on your computer: the images are built by AWS CodeBuild, and the trials run on AgentCore.

This cell sets the settings of the notebook and defines some helpers. `p2()` runs the runner as a child process, prints its output while it runs, and prints a line every 30 seconds when it is quiet (a build has no output for some minutes). The helpers hide 12-digit numbers (account IDs) and your home directory.

`PREFIX` is the name prefix of every resource. A different prefix gives you a second, independent copy. To reuse the images of the README smoke test, set it to `evals-p2b-`.

In [ ]:
import json, os, queue, re, signal, subprocess, sys, threading, time
from pathlib import Path
from IPython.display import Markdown, display

PREFIX = os.environ.get("P2_PREFIX", "evals-nb-")   # every resource name starts with this
AWS_PROFILE = None                                  # for example "my-sandbox". None = the default credential chain
REGION = "us-east-1"

HERE = Path.cwd()
MODULE = HERE.parent
if not ((MODULE / "model_config.py").exists() and (MODULE / "datasets").is_dir()):
    raise RuntimeError("Open this notebook from the agentcore folder of the Coding Assistant module.")
sys.path.insert(0, str(MODULE))
import tomllib

if AWS_PROFILE:
    os.environ["AWS_PROFILE"] = AWS_PROFILE
ENV = {**os.environ, "P2_PREFIX": PREFIX, "P2_MODULE": str(MODULE), "PYTHONUNBUFFERED": "1"}

def read_json(path):
    return json.loads(Path(path).read_text())

def read_toml(path):
    return tomllib.loads(Path(path).read_text())

def show_tree(root):
    root = Path(root)
    print(root.name + "/")
    for p in sorted(root.rglob("*")):
        if "__pycache__" not in p.parts:
            print("    " * len(p.relative_to(root).parts) + p.name + ("/" if p.is_dir() else ""))

def mask(text):
    return re.sub(r"(?<!\d)\d{12}(?!\d)", "ACCT", text).replace(str(Path.home()), "~")

def table(rows, headers):
    display(Markdown("\n".join(["| " + " | ".join(headers) + " |", "|" + "---|" * len(headers)]
                               + ["| " + " | ".join(str(c) for c in r) + " |" for r in rows])))

def stream(cmd, until=None, beat=30, tick=None, check=True):
    # Run a command, print its output as it comes, return (exit code, lines). `until`: stop printing at the first line that matches.
    t0, lines, quiet = time.time(), [], False
    proc = subprocess.Popen(cmd, cwd=HERE, env=ENV, stdin=subprocess.DEVNULL, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    q = queue.Queue()
    threading.Thread(target=lambda: ([q.put(l) for l in proc.stdout], q.put(None)), daemon=True).start()
    try:
        while True:
            try:
                line = q.get(timeout=beat)
            except queue.Empty:
                print(f"   ... still running, {time.time() - t0:.0f} s" + (tick() if tick else ""), flush=True)
                continue
            if line is None:
                break
            line = mask(line.rstrip())
            lines.append(line)
            quiet = quiet or bool(until and re.search(until, line))
            if not quiet:
                print(line, flush=True)
    except KeyboardInterrupt:                        # ask the runner to stop its sessions, then stop waiting
        proc.send_signal(signal.SIGINT)
        try:
            proc.wait(60)
        except subprocess.TimeoutExpired:
            proc.kill()
        raise
    rc = proc.wait()
    print(f"[exit {rc}, {time.time() - t0:.0f} s]")
    if check and rc != 0:
        raise RuntimeError("The command failed. Read the last lines above.")
    return rc, lines

def p2(*args, **kw):
    return stream([sys.executable, "-u", "p2_v2.py", "--prefix", PREFIX, *args], **kw)

print(f"prefix {PREFIX}, module {MODULE.name}, working folder {HERE.name}")

### Check Python, boto3, and the AWS identity

The cell shows the name of your role or user and hides the account ID. If a check fails, it tells you what to do.

In [ ]:
import importlib.metadata as md
import boto3

def version(v):
    return tuple(int(x) for x in re.findall(r"\d+", v)[:3])

print(f"{'OK ' if sys.version_info >= (3, 11) else 'ERR'} Python {sys.version.split()[0]} (need 3.11 or later)")
ok_boto = version(md.version("boto3")) >= (1, 44, 7)
print(f"{'OK ' if ok_boto else 'ERR'} boto3 {md.version('boto3')} (need 1.44.7 or later)")
if not ok_boto:
    raise RuntimeError("Run `%pip install -U 'boto3>=1.44.7'` in a cell, restart the kernel, and run this cell again.")

session = boto3.Session()
try:
    who = session.client("sts", region_name=REGION).get_caller_identity()
except Exception as e:
    raise RuntimeError(f"No usable AWS credentials ({type(e).__name__}). Run `aws sso login` or set AWS_PROFILE in the settings cell.") from None

kind, _, rest = who["Arn"].split(":", 5)[5].partition("/")
name = rest.split("/")[0] if kind == "assumed-role" else rest.split("/")[-1]
print(f"OK  AWS identity: {kind} {name}, account ************, Region {REGION} (fixed by the runner)")
ACCOUNT_ID, CALLER_ARN, CALLER_NAME, CALLER_KIND = who["Account"], who["Arn"], name, kind   # never printed

### Check your permissions before anything is created

Bootstrap makes IAM roles and many other resources. If your role cannot do that, you want to know now, not after a 6 minute build. The first part of the cell calls one read-only API of each service, which shows a missing permission or a blocking policy. The second part asks the IAM policy simulator about the write actions (it needs `iam:SimulatePrincipalPolicy`; if you do not have it, the cell says so and relies on the first part). The simulator checks your identity policies, but not service control policies, so a pass here is not a promise.

The full list of actions is in section 2 of [TESTER-GUIDE.md](TESTER-GUIDE.md).

In [ ]:
def c(service):
    return session.client(service, region_name=REGION)

reads = {
    "iam": lambda: c("iam").list_roles(MaxItems=1),
    "ecr": lambda: c("ecr").describe_repositories(maxResults=1),
    "s3": lambda: c("s3").list_buckets(),
    "codebuild": lambda: c("codebuild").list_projects(),
    "logs": lambda: c("logs").describe_log_groups(limit=1),
    "bedrock-agentcore-control": lambda: c("bedrock-agentcore-control").list_agent_runtimes(maxResults=1),
    "resourcegroupstaggingapi": lambda: c("resourcegroupstaggingapi").get_resources(ResourcesPerPage=1),
}
problems = []
for service, call in reads.items():
    try:
        call(); print(f"OK  read  {service}")
    except Exception as e:
        problems.append(f"{service}: {type(e).__name__}"); print(f"ERR read  {service}: {type(e).__name__}")

WRITES = ["iam:CreateRole", "iam:PutRolePolicy", "iam:PassRole", "iam:DeleteRole", "sts:AssumeRole", "ecr:CreateRepository",
          "s3:CreateBucket", "s3:PutObject", "codebuild:CreateProject", "codebuild:StartBuild", "logs:CreateLogGroup",
          "bedrock-agentcore:CreateAgentRuntime", "bedrock-agentcore:InvokeAgentRuntimeCommand"]
try:
    arn = CALLER_ARN if CALLER_KIND == "user" else c("iam").get_role(RoleName=CALLER_NAME)["Role"]["Arn"]
    sim = c("iam").simulate_principal_policy(PolicySourceArn=arn, ActionNames=WRITES)["EvaluationResults"]
    denied = [r["EvalActionName"] for r in sim if r["EvalDecision"] != "allowed"]
    print(f"{'OK ' if not denied else 'ERR'} simulator: {len(WRITES) - len(denied)} of {len(WRITES)} write actions allowed")
    problems += [f"not allowed by your policies: {a}" for a in denied]
except Exception as e:
    print(f"--  simulator skipped ({type(e).__name__}); the read checks above are all that this cell can say")

if problems:
    raise RuntimeError("Fix these before you continue (see section 2 of TESTER-GUIDE.md): " + "; ".join(problems))
print("\nPermission pre-check passed.")

### The models and the agent

All model IDs of the module are in one file, `../model_config.py`. The runner reads it, so a change there reaches every trial.

| Name | Role |
|---|---|
| `CODEX_MODEL_ID` | The model of the **agent under test**. Codex calls it through the Bedrock endpoint `bedrock-mantle.<region>.api.aws` |
| `JUDGE_MODEL_ID` | The **PR review judge**: it decides if an agent finding and an oracle finding describe the same defect |
| `VERIFIER_MODEL_ID` | The **coding grader**: the validation agent, the validation judge, the task rubric judge, and the taste judge |
| `CLAUDE_CODE_MODEL_ID` | A second agent. This runner does not use it |

**Why the judges are a different model from the agent.** A model that grades its own work tends to like it: it shares the blind spots of the answer that it grades. A judge from a different model family gives a second opinion. A small, fast judge model also keeps the many judge calls cheap. And the split lets you give each phase only its own model: in section 3 you see that the agent role can call the agent model only, and the grader role the judge model only. The agent cannot call the judge, and the grader cannot call the agent model.

**What Codex is.** [Codex](https://github.com/openai/codex) is an open-source coding agent that runs in a terminal. It reads files, runs shell commands, and edits code until it thinks that the task is done. The runner starts it in the session as `codex exec` with the task instruction, and with these settings:

- `../codex-bedrock.toml` becomes the Codex configuration (`$CODEX_HOME/config.toml`). It sets the model provider `amazon-bedrock` and the Region, and turns off update checks, analytics, and feedback. Codex needs no API key: it signs its calls with the AWS credentials of the agent phase.
- `--model CODEX_MODEL_ID` from `model_config.py`, `-c web_search=disabled` (the firewall also blocks it), and `--json`, so that each step of the agent is one JSON line in `agent/codex.txt`.
- `--dangerously-bypass-approvals-and-sandbox`: Codex does not ask before a command, and it does not add its own sandbox. The session firewall is the sandbox.

The task files allow only the Bedrock endpoints of one Region, and the Codex configuration names one Region. The cell checks that they agree with the Region of the runner.

In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location("model_config", MODULE / "model_config.py")
models = importlib.util.module_from_spec(spec); spec.loader.exec_module(models)
for name in ("CODEX_MODEL_ID", "JUDGE_MODEL_ID", "VERIFIER_MODEL_ID"):
    print(f"{name:<18} {getattr(models, name)}")

codex_config = (MODULE / "codex-bedrock.toml").read_text()
print("\n--- codex-bedrock.toml (without comments) ---")
print("\n".join(l for l in codex_config.splitlines() if l.strip() and not l.lstrip().startswith("#")))

regions = {read_toml(MODULE / "codex-bedrock.toml")["model_providers"]["amazon-bedrock"]["aws"]["region"]}
for toml in (MODULE / "datasets").glob("*/tasks/*/task.toml"):
    cfg = read_toml(toml)
    for host in cfg["environment"].get("allowed_hosts", []) + cfg["verifier"].get("allowed_hosts", []):
        regions.add(host.split(".")[1])
assert regions == {REGION}, f"The task files and codex-bedrock.toml must all use {REGION}, not {sorted(regions)}"
print(f"\nOK  codex-bedrock.toml and every task.toml use {REGION}")

## 3. Bootstrap: Create the Resources

`bootstrap` creates what a fresh account lacks. It is idempotent: each step checks if the resource exists, so you can run it again at any time. It takes about 15 seconds.

| Resource | Name | Purpose |
|---|---|---|
| ECR repository | `<prefix>fw` | Holds one image for each task, tagged with the task name |
| S3 bucket | `<prefix>src-<account>-us-east-1` | Holds the build contexts. Private, and objects expire after 3 days |
| Log group | `/aws/codebuild/<prefix>build` | The build logs, kept for 3 days |
| IAM role | `<prefix>codebuild-role` | Lets CodeBuild read the context and push to the repository |
| IAM role | `<prefix>grader-role` | The AgentCore execution role: the credentials of the **grader phase** (the judge model only) |
| IAM role | `<prefix>agent-role` | The credentials of the **agent phase** (the Codex model only). The runner assumes it and hands the temporary credentials to the agent |
| CodeBuild project | `<prefix>build` | Builds the arm64 images |

**Tags and prefix.** Each resource has the tags `Project=evals-streamline-spike` and `Expiry=<date two weeks out>`. Nothing deletes a resource on that date: the tag only tells you, and your colleagues, that it is a temporary test. The name prefix is what the cleanup uses. It never touches a name that does not start with your prefix.

In [ ]:
rc, out = p2("bootstrap", until=r"^\{$")   # the runner prints its report as JSON at the end: the next cell shows it as a table

In [ ]:
report = read_json(HERE / "jobs" / "bootstrap-report.json")
table([(s["step"], s["action"], s["secs"]) for s in report["steps"]], ["step", "result", "seconds"])
print(f"bootstrap took {report['secs']} s. IAM changes need some seconds to reach the other services; "
      f"the runner retried {sum(p['retries'] for p in report['propagation'].values())} times to wait for them.")

### Look at the permissions that the roles have

The two roles that matter for the seal are narrow on purpose. This cell prints the actions and resources of each policy that `bootstrap` installed (`p2_v2.py policies` prints the full documents). Notice that the agent role can call the Codex model and nothing else, and that the grader role can call the judge model and nothing else.

In [ ]:
rc, out = p2("policies", until=r"^###")
docs = re.split(r"^### ", "\n".join(out), flags=re.M)[1:]
for doc in docs:
    title, _, body = doc.partition("\n")
    print(f"\n{title}")
    for st in json.loads(body)["Statement"]:
        actions = st["Action"] if isinstance(st["Action"], list) else [st["Action"]]
        res = st["Resource"] if isinstance(st["Resource"], list) else [st["Resource"]]
        print(f"   {st['Effect']}: {', '.join(actions)[:110]}\n      on {', '.join(res)[:110]}")

## 4. The Datapoints and What Grades Them

### 4.1 The task format

Each example should be in a standard format. Then a runner can run any agent on any example, and other people can run your eval again and get the same result. The module keeps each datapoint as one **task directory**. Many benchmarks use this layout.

```text
tasks/pr-1234-reject/
├── task.toml          # settings for the agent and the grader, and metadata about the datapoint
├── instruction.md     # the instruction that the agent reads
├── environment/
│   └── Dockerfile     # the build file of the agent's image: rolls back to the parent commit, seals git, installs dependencies
├── tests/             # grader only
│   ├── test.sh        # the grader entry point, always writes a reward file
│   ├── score.py       # (PR review) compares the agent output to the oracle
│   └── oracle.json    # (PR review) the oracle
└── solution/          # grader only
    ├── solve.sh       # submits the oracle as the agent output, to test the grader
    └── oracle.patch   # (coding) the merged fix
```

The most important part of this format is the split between the agent and the grader:

- **The agent sees** `instruction.md` and the image that `environment/` builds. For a PR review, the instruction is the same for all tasks. It does not name the PR or the answer.
- **Only the grader sees** `tests/` and `solution/`. In this runner, `tests/` is in the image but readable only by root, and it is copied to `/tests` after the agent's processes are killed. `solution/` never goes into the image.

`task.toml` has these parts:

| Table | Fields | Meaning |
|---|---|---|
| `[task]` | `name`, `description` | The name of the task, for example `awslabs/mcp__pr4615-reject` |
| `[agent]` | `timeout_sec` | How long the agent may work |
| `[environment]` | `network_mode = "allowlist"`, `allowed_hosts` | What the agent may reach: the Bedrock endpoints of the agents, never GitHub. This runner runs Codex, so it opens only the Codex endpoint |
| `[verifier]` | `timeout_sec`, `environment_mode`, `network_mode`, `allowed_hosts` | How the grader runs. A PR review grader needs only the findings file (`separate`); a coding grader must run the agent's code, so it runs in the agent's image after the agent stops (`shared`), with only the Bedrock runtime endpoint for the judges |
| `[verifier.env]` | `JUDGE_MODEL_ID` or `VERIFIER_MODEL_ID`, AWS settings | What the grader gets. The runner fills in the model from `model_config.py` and the grader-role credentials |
| `[metadata.*]` | origin, expected verdict, severities, risk, hidden tests, cause and fix | Facts about the datapoint for you and the grader. The agent never sees `task.toml` |

**Shared grader files.** The grader code that is the same in each task of a dataset (for example `score.py`, `tests/lib/*.py`) has one source in `../datasets/_shared/`. Each task must be complete in its own directory, because only that directory goes into the image. So `datasets/_shared/sync.py` copies the shared files into each task. The cell checks that the copies match the source.

In [ ]:
REVIEW_DIR = MODULE / "datasets" / "awslabs-mcp-pr-review" / "tasks"
CODING_DIR = MODULE / "datasets" / "strands-evals-coding" / "tasks"
TASK_DIRS = {"pr4121": REVIEW_DIR / "awslabs__mcp__pr4121-approve", "pr4615": REVIEW_DIR / "awslabs__mcp__pr4615-reject",
             "pr371": CODING_DIR / "strands-agents__evals__pr371-fix", "pr338": CODING_DIR / "strands-agents__evals__pr338-fix"}
REVIEW_TASK, APPROVE_TASK, CODING_TASK = TASK_DIRS["pr4615"], TASK_DIRS["pr4121"], TASK_DIRS["pr371"]

show_tree(REVIEW_TASK)
print()
show_tree(CODING_TASK)

spec = importlib.util.spec_from_file_location("sync", MODULE / "datasets" / "_shared" / "sync.py")
sync = importlib.util.module_from_spec(spec); spec.loader.exec_module(sync)
stale = sync.stale_copies()
print(f"\nshared grader files: {len(sync.pairs())} copies, {len(stale)} differ from datasets/_shared/"
      + ("" if not stale else ". Run `python datasets/_shared/sync.py`: " + ", ".join(p.name for p in stale)))

In [ ]:
for name, task in (("PR review", REVIEW_TASK), ("coding", CODING_TASK)):
    cfg = read_toml(task / "task.toml")
    env, ver = cfg["environment"], cfg["verifier"]
    print(f"{name}: {cfg['task']['name']}")
    print(f"   agent:  timeout {cfg['agent']['timeout_sec']:.0f} s, network {env['network_mode']} {env['allowed_hosts']}")
    print(f"   grader: {ver.get('environment_mode', 'shared')}, timeout {ver['timeout_sec']:.0f} s, "
          f"hosts {ver.get('allowed_hosts', 'any (a separate grader)')}, env {sorted(ver.get('env', {}))}")
    print(f"   metadata tables: {sorted(cfg['metadata'])}")
    assert env["network_mode"] == "allowlist" and not any("github" in h for h in env["allowed_hosts"]), "The agent must not reach GitHub"

### 4.2 How the PR review datapoints were made

The PR review datapoints come from the PR history of [awslabs/mcp](https://github.com/awslabs/mcp). We made them before the workshop with [sample-code-eval-harvester](https://github.com/aws-samples/sample-code-eval-harvester), a CLI that a coding agent drives: the CLI collects the facts from git and GitHub, and the agent makes the decisions. The steps are: survey the PRs that can become datapoints; capture each PR (the review iterations, the patches, the comments, and the reviews); label each review comment; write the findings with a severity and a risk level for the change; emit a sealed task directory; and verify that the datapoint is correct and that the agent cannot see the answer.

Each review comment gets one label. Only defects go into the oracle.

| Label | Meaning | In the oracle |
|-------|---------|---------------|
| **Defect** | The reviewer blocks the PR for this problem | Yes, with severity `low`, `medium`, or `high` |
| **Nit** | Style or preference | No |
| **Question** | A request for information | No |
| **Approval** | A sign-off with no finding | No |
| **Bot** | A comment from an automated tool | No |

> **Note:** Do not label a nit as a defect. If you do, the eval rewards an agent that reports nits.

### 4.3 The two PR review datapoints

The dataset has a reject datapoint and an approve datapoint. Both changes are security-relevant.

| | Reject datapoint | Approve datapoint |
|---|---|---|
| PR | [#4615](https://github.com/awslabs/mcp/pull/4615) | [#4121](https://github.com/awslabs/mcp/pull/4121) |
| Server | `mysql-mcp-server` | `eks-mcp-server` |
| Change | Read-only mode also rejects SQL statements that start with `DO`, `START`, `SHUTDOWN`, and other mutating verbs | The Kubernetes client accepts EKS cluster CAs that have no Authority Key Identifier |
| Version under review | Iteration 0. A reviewer requested changes on this version | Iteration 1, the final version. Two reviewers approved it |
| Expected verdict | `block` | `approve` |
| Oracle | 1 high, 2 medium, and 3 low findings | No findings |

**PR 4615 (reject).** The PR tries to close a bypass of the read-only guard. A reviewer found that the bypass is still open: a MySQL comment such as `#x` before the verb hides the verb from the new check. This is a clear high-severity defect, and a good review agent must find it.

**PR 4121 (approve).** EKS makes cluster CAs without an Authority Key Identifier, and OpenSSL 3 rejects them when the `VERIFY_X509_STRICT` flag is set. The fix removes only that flag. Certificate verification stays required (`CERT_REQUIRED`). The change edits TLS settings, so it looks dangerous: an agent that blocks each change that looks dangerous fails this datapoint. An agent that blocks all PRs gets a perfect score on reject datapoints, and the approve datapoint finds this problem.

**Make sure that an approval is correct.** A human approval does not prove that the code is correct. Before you use an approved PR as an approve datapoint, examine the later commits to the same files. If a later commit fixes the same code, the approval was probably wrong. An earlier version of this dataset used [#4268](https://github.com/awslabs/mcp/pull/4268) as the approve datapoint. Codex blocked it, and ten weeks later [#4701](https://github.com/awslabs/mcp/pull/4701) closed the gap that Codex had found. The agent was correct, and the datapoint was wrong. For PR 4121, no later commit changes the code.

The candidate files in `datasets/awslabs-mcp-pr-review/candidates/` record what was captured and decided for each PR. This cell shows a summary.

In [ ]:
from collections import Counter
CANDIDATES = MODULE / "datasets" / "awslabs-mcp-pr-review" / "candidates"
for pr in (4615, 4121):
    cand = read_json(CANDIDATES / f"pr-{pr}.json")
    risk = cand["change_risk"]
    print(f"PR {pr}: {len(cand['iterations'])} iterations, reviews {dict(Counter(v['state'] for v in cand['review_verdicts']))}")
    print(f"   comment labels: {dict(Counter(c['classification'] for c in cand['comments']))}")
    print(f"   findings: {dict(Counter(f['severity'] for f in cand['findings'])) or 'none'}, "
          f"risk structural {risk['risk_structural']}, classified {risk['risk_classified']}")

### The oracle and the instruction

The oracle (`tests/oracle.json`) is the correct answer. Each finding is a defect, not a nit. It has a severity, and only `high` findings block a merge. Its location is a span of lines that touches a line that the change adds: the agent can find only what the change contains. An agent finding passes the location step if its line is in a span of the oracle finding. A span that is too narrow is unfair, because a correct finding on the next line gets no credit. Thus the high finding of the reject task has two spans: the function that strips the comments and runs the scan, and the definition of the statement-start pattern. A `reference_only` finding is a real defect that the agent does not have to find: it counts in the overall coverage, not in the required coverage.

The instruction must be the same for all review tasks, and it must not refer to the PR, the reviewer, or the findings. If it does, the agent can learn the answer from the instruction. It tells the agent to write its findings to `/logs/artifacts/findings.json`.

In [ ]:
oracle = read_json(REVIEW_TASK / "tests" / "oracle.json")
print("expected verdict:", oracle["expected_verdict"], "\n")
for f in oracle["findings"]:
    spans = ", ".join(f"{l['line_start']}-{l['line_end']}" for l in f["locations"])
    print(f"  [{f['severity']:<6}] {f['locations'][0]['path'].split('/')[-1]}:{spans}{' (reference only)' if f['reference_only'] else ''}")
    print(f"           {f['statement'][:100]}...")
instructions = {(t / "instruction.md").read_text() for t in (REVIEW_TASK, APPROVE_TASK)}
assert len(instructions) == 1, "The review instruction must be the same for all tasks"
print("\n--- what the agent reads (instruction.md, first 12 lines; the same for both review tasks) ---")
print("\n".join((REVIEW_TASK / "instruction.md").read_text().splitlines()[:12]))

**Severity and risk are different.** Severity tells how bad a finding is. Risk tells how dangerous the change is. `datasets/awslabs-mcp-pr-review/rubric.md` tells the labeler how to set a severity: it starts from a standard review rubric and adds the conventions of awslabs/mcp, from files such as `DESIGN_GUIDELINES.md` and `CONTRIBUTING.md`. These conventions are the rules of the team, which public benchmarks do not have. `risk-map.toml` maps path prefixes to a **structural** risk (a path that is not in the map is `medium`), and a model or a person sets the **classified** risk from the content of the change. For PR 4121 the two disagree (medium and high), and `task.toml` records `change_risk_disagreement = true`. Use risk to decide if an agent can approve low-risk PRs without a human, and examine the low-risk results alone before you do. This dataset has no low-risk datapoint, so it cannot tell you that.

### 4.4 How a PR review is scored

For each task, the agent writes a list of findings. `tests/score.py` compares them with the oracle in two steps. First, it compares the file and the line. Then an LLM judge (`JUDGE_MODEL_ID`) decides if the two findings describe the same problem. An LLM judge can give different answers to the same question, so `score.py` makes the decision stable in two ways:

- If the agent statement is the same as the oracle statement (case and whitespace do not matter), it is a match, with no judge call.
- Otherwise, the judge asks the model 3 times (`votes` in `tests/judge.toml`), and the majority decides. `verifier/judge.json` records each answer. A split vote (2 yes and 1 no) shows a finding near the line between a match and no match. Examine it.

`score.py` writes these signals to `reward.json`:

| Signal | Question |
|--------|----------|
| `coverage_required` | What fraction of the high (blocking) findings did the agent find? |
| `coverage_all` | What fraction of all oracle findings did the agent find? |
| `precision_adjudicated` | What fraction of the agent findings did the judge match to an oracle finding? |
| `precision_strict` | What fraction of the agent findings have the same location as an oracle finding? No judge |
| `uncredited` | How many agent findings match no oracle finding? |
| `credited_<severity>`, `oracle_<severity>` | How many findings of each severity did the agent find, and how many are in the oracle? |

If the agent reports no findings, precision has no answer. That is the correct result on an approve task, so a value of 0 would be wrong. `score.py` leaves the precision keys out of `reward.json` and writes them as `null` in `reward_details.json`. If `precision_strict` and `precision_adjudicated` are very different, the score depends much on the judge: examine the judge.

**The verdict and pass or fail.** The instruction tells the agent to request changes only for a high-severity defect. Thus the predicted verdict comes from the findings: `block` if a finding has the severity `high`, and `approve` if not. A review **passes** if the verdict is right and, on a reject task, `coverage_required` is 1.0. Over a dataset, calculate these metrics (section 9 does):

| Metric | Question |
|--------|----------|
| **Accuracy** | What fraction of trials did the agent pass? |
| **Block recall** | Of the PRs that must be blocked, what fraction did the agent block? |
| **False approve rate** | How often did the agent approve a bad PR? This error has the highest cost |
| **False block rate** | How often did the agent block a good PR? |

Two rules: an agent finding that the reviewer did not make is **uncredited**, not wrong, because reviewers do not find all defects. And keep each signal: the signals show why a task failed.

### 4.5 The coding datapoint

The coding datapoints come from bug fixes in [strands-agents/evals](https://github.com/strands-agents/evals). A PR review datapoint asks: *does the agent find the defects in this change?* A coding datapoint asks: *can the agent make the change?*

| Part | PR review datapoint | Coding datapoint |
|------|--------------------------|-----------------------|
| Start state | The base commit, with the change applied | The base commit (the parent of the fix), with no change |
| Instruction | The same for all tasks: review the change | An issue report, different for each task |
| Agent output | A list of findings | Changes to the code |
| Oracle | The defects that the reviewers found | The fix that the team merged |
| Grader | An LLM judge matches the findings to the oracle | A reward stack: tests, a validation agent, a rubric judge, and a taste judge |

**How the datapoints were made.**

1. **Select merged PRs that fix a bug.** Each PR must close an issue that describes the problem, and its tests must run without a network or a model.
2. **Write the instruction from the issue.** Keep the symptom and the expected behavior. Remove the root cause, the file names, and the function names. The agent must find the cause itself.
3. **Write the hidden tests.** Do not copy the PR tests: they often import private functions, or check one design of the fix. Write tests that use only the public API, and that check only the behavior that the instruction asks for. A different correct fix must pass.
4. **Write the validation spec and the rubric.** The spec has user stories, a procedure for each story, and several cases (inputs and expected values). The rubric has fail-to-pass and pass-to-pass criteria.
5. **Check every mechanism on the oracle and on an empty agent** (section 7).
6. **Build a sealed environment.** Clone only the base commit, and install the dependencies. The agent cannot find the fix in the git history, and it can run the tests.

The task in this notebook is PR [#371](https://github.com/strands-agents/evals/pull/371) (issue #370). In a flat multi-agent trace, a tool-level evaluator gives false negatives: the judge thinks that the user asked only a part of the question, because the extractor picks a sub-agent as the root agent. The merged fix is one function, 7 lines, but the agent must find which function causes the symptom. The module has a second, larger coding task, `pr338`, that this notebook does not run. `task.toml` records the cause and the merged fix. The agent never sees `task.toml`.

In [ ]:
meta = read_toml(CODING_TASK / "task.toml")["metadata"]
origin, verifier, story = meta["origin"], meta["verifier"], meta["narrative"]
print(f"origin:  {origin['repo']} PR {origin['pr_numbers']}, issue {origin['issue_numbers']}, base commit {origin['base_commit'][:9]}")
print(f"tests:   {verifier['fail_to_pass']} fail-to-pass and {verifier['pass_to_pass']} pass-to-pass hidden tests")
print("cause:  ", " ".join(story["description"].split()))
print("fix:    ", " ".join(story["solution"].split()))
print("\n--- what the agent reads (instruction.md, first 14 lines) ---")
print("\n".join((CODING_TASK / "instruction.md").read_text().splitlines()[:14]))

### The reward stack

A fix that passes the tests is not always a fix that a maintainer would merge. So the grader grades the fix as [Senior SWE-Bench](https://senior-swe-bench.snorkel.ai/blog/2026-06-16-how-it-works) does. The hidden tests are in two groups:

- **Fail to pass (f2p).** These tests fail on the base commit and pass with the merged fix. They show that the agent fixed the bug.
- **Pass to pass (p2p).** These tests pass on the base commit and must still pass. They show that the agent did not break something. The grader also runs the full unit test suite of the repository (`existing_suite`).

| Mechanism | What it checks | LLM? | Gate for a tasteful solve |
|-----------|----------------|------|---------------------------|
| **Hidden tests** | f2p, p2p, and the existing suite. The agent's changes to test files are undone first, so the agent cannot change a test so that it passes | No | All pass |
| **Validation agent** | Tests that an agent writes from an expert *validation spec*, for the shape of the agent's change. A validation judge reviews the tests | Yes | All stories pass |
| **Task rubric judge** | Task-specific criteria that tests cannot check well | Yes | Score > 0.5 |
| **Taste judge** | Ten yes/no checks: five against the merged fix (minimality, approach, hygiene, fluency, craftsmanship) and five for fit with the codebase (style, patterns, libraries, abstraction, documentation) | Yes | At least 3 of 5 in each group |
| **Patch bloat** | Source lines of the agent's change divided by those of the merged fix | No | < 2 |
| **Agent-written tests** | Did the agent add a test that fails on the base and passes on its fix? | No | Reported only |

Why both hidden tests and a validation agent? A pre-written test can check only the interface that its author expected. The validation agent reads the agent's change and writes the test for the shape that the agent built. Why yes/no taste checks and no 1 to 5 scale? LLM judges that score on a scale tend to give 4 or 5 to everything, and a yes/no question is easier to check against a human.

**How the reward is formed.** `tests/test.sh` runs six stages after the agent stops, and `tests/lib/aggregate.py` combines them:

| Stage | What it does | Files in `verifier/` |
|---|---|---|
| 0 | Save the agent's change as a patch | `agent.patch` |
| 1 | Restore the base test files, add the hidden tests, run them and the existing suite | `junit.xml`, `pytest.log` |
| 2 | Run the agent's own tests on its fix and on the base | `agent_tests.json` |
| 3 | Validation agent and validation judge | `validation.json` |
| 4 | Rubric judge, taste judge, patch bloat | `rubric.json`, `taste.json`, `bloat.json` |
| 5 | Combine the signals | `reward.json`, `reward_details.json` |

`correct` is 1 when the hidden tests, the existing suite, and every validation story pass. `reward` (the same as `tasteful_solve`) is 1 only if `correct` is 1 **and** the rubric score is above 0.5, at least 3 of 5 taste checks pass in each group, and the patch bloat is less than 2. The resolved rate of an agent is the mean `correct`; the tasteful-solve rate is the mean `reward`. The difference tells you how often the agent writes a fix that works but that a reviewer would not merge. The rubric does not gate a solve alone: an empty change fulfills the pass-to-pass criteria ("no interface change") and gets some rubric credit.

If an LLM stage fails (for example, throttling), or the validation judge discards the validation tests, the trial is **invalid**: there is no `reward.json`, and the trial is not counted. An infrastructure failure is not a failure of the agent.

In [ ]:
print("tests/ of the coding task (the grader only):")
for p in sorted((CODING_TASK / "tests").rglob("*")):
    if p.is_file() and "__pycache__" not in p.parts:
        print("  ", p.relative_to(CODING_TASK / "tests"))
print("\n--- the merged fix, the oracle of this task (solution/oracle.patch) ---")
print((CODING_TASK / "solution" / "oracle.patch").read_text()[:1800])

In [ ]:
spec_toml = read_toml(CODING_TASK / "tests" / "validate" / "validation_spec.toml")
rubric = read_toml(CODING_TASK / "tests" / "judge" / "rubric.toml")
for story in spec_toml["stories"]:
    print(f"story {story['id']}: {' '.join(story['description'].split())[:220]}")
    print(f"   {len(story['procedure'])} procedure steps, {len(story.get('cases', []))} cases")
print("\nrubric criteria:")
for c in rubric["criteria"]:
    print(f"   [{c['kind']}] {c['id']}: {c['text'][:120]}")

### 4.6 The image build file and the git seal

`environment/Dockerfile` is the build file of the image that the agent works in. For the coding task it does these steps:

1. Clones only the base commit, the parent of the fix.
2. Removes the remote and the remote refs, expires the reflog, and runs garbage collection. After this step, only one commit is in the repository.
3. Writes the base commit to `/var/lib/task_base_ref`. The grader uses it to find the changes of the agent.
4. Installs Python and the test dependencies. `requirements.lock` pins each package to the newest version on the day that the fix was merged, so a new version of a dependency cannot break a test later.
5. **Runs the full unit test suite at build time.** If a test fails on the base, the build fails. Then a failure in a trial is caused by the agent, not by the environment.
6. Installs Codex. The agent has no package index at run time, so all software is installed at build time.

A PR review build file does the same, and applies `change.patch` as uncommitted changes: the agent reviews those changes. The runner uses the task's build file unchanged, and adds a small layer for the firewall (`image/layer.Dockerfile`). The cell prints the build steps without the comments.

In [ ]:
lines = (CODING_TASK / "environment" / "Dockerfile").read_text().splitlines()
print("\n".join(l for l in lines if l.strip() and not l.lstrip().startswith("#"))[:2600])

### 4.7 Leakage scan

Scan all files that the agent sees (`instruction.md` and `environment/`). Search for the PR numbers, the reviewer names, text from the review comments, the merge commit, the name of the hidden test file, and lines that the fix adds. If the agent can see this data, the score measures search, not skill.

First, test the scan: put a canary token in a temporary copy of the task, and make sure that the scan finds it. If it does not, the scan has a defect. The copy is temporary, so the task files never change.

In [ ]:
import shutil, tempfile

def agent_visible(task):
    return [task / "instruction.md", *(p for p in sorted((task / "environment").rglob("*")) if p.is_file())]

def scan(task, needles):
    return [(str(p.relative_to(task)), n) for p in agent_visible(task) for n in needles
            if n and n.lower() in p.read_text(errors="ignore").lower()]

def leakage_scan(task, needles, canary="CANARY-7f3a91"):
    with tempfile.TemporaryDirectory() as tmp:          # test the scan itself on a copy
        copy = Path(shutil.copytree(task, Path(tmp) / task.name))
        (copy / "environment" / "canary.txt").write_text(canary)
        assert scan(copy, [canary]), "The scan did not find the canary. The scan has a defect."
    return scan(task, needles)

for pr, task in ((4615, REVIEW_TASK), (4121, APPROVE_TASK)):
    cand = read_json(CANDIDATES / f"pr-{pr}.json")
    needles = [f"#{pr}", f"pull/{pr}", *{v["author"] for v in cand["review_verdicts"]},
               *(c["body"][:60] for c in cand["comments"] if len(c["body"]) >= 60)]
    print(f"{task.name}: {len(needles)} needles, leaks: {leakage_scan(task, needles) or 'none'}")

meta = read_toml(CODING_TASK / "task.toml")["metadata"]
needles = [f"#{n}" for n in meta["origin"]["pr_numbers"] + meta["origin"]["issue_numbers"]]
needles += [meta["origin"]["merge_commit"][:12], Path(meta["verifier"]["hidden_tests"]).stem]
needles += [l[1:].strip() for l in (CODING_TASK / "solution" / "oracle.patch").read_text().splitlines()
            if l.startswith("+") and not l.startswith("+++") and len(l.strip()) >= 40]
print(f"{CODING_TASK.name}: {len(needles)} needles, leaks: {leakage_scan(CODING_TASK, needles) or 'none'}")

### Where the seal is in an AgentCore session

| Path to the answer | Control | Where you can see it |
|---|---|---|
| **Git:** `git log --all` shows the fix | The task's build file clones one commit and removes remotes and the reflog | Section 4.6, and the check in section 5 |
| **Network:** the agent looks up the fix on GitHub | An empty network namespace, with an allowlist proxy as its only way out | Section 8 |
| **Grader:** the agent reads or changes the hidden tests | The tests are in the image as root-only files. They are copied to `/tests` only after the agent's processes are killed, and the base test files are restored before the hidden tests run | The check in section 5, and `phase_switch` in the timings of section 6 |

## 5. Build and Deploy the Images

Each task needs an image and a runtime. **Build** sends the task's build file and the firewall layer to CodeBuild, which builds an arm64 image and pushes it to your ECR repository. **Deploy** creates an AgentCore runtime that starts sessions from that image.

The build is the slow step. The build file clones the repository, installs the dependencies, and runs the full test suite of the repository, so a build takes **2 to 8 minutes** (`pr4121` and `pr4615` 2 to 3 minutes, `pr371` 6 to 8 minutes). The images build at the same time, so the wait is the longest build. The runner prints `<task> build started` and then nothing until the build ends. The cell prints a line every 30 seconds with the phase of each build.

Both cells skip what exists, so you can run them again after an interruption. Set `REBUILD = True` to build the images again. A build costs about $0.12 for each image.

In [ ]:
TASKS = ["pr4121", "pr4615", "pr371"]     # the tasks of this notebook (the module also has pr338)
REBUILD = False
ecr, cb = session.client("ecr", region_name=REGION), session.client("codebuild", region_name=REGION)
ctl = session.client("bedrock-agentcore-control", region_name=REGION)

def runtime_name(task):
    return re.sub(r"[^A-Za-z0-9_]", "_", PREFIX) + task

def image_exists(task):
    try:
        ecr.describe_images(repositoryName=PREFIX + "fw", imageIds=[{"imageTag": task}]); return True
    except Exception:
        return False

def runtime_status(task):
    for r in ctl.list_agent_runtimes(maxResults=100)["agentRuntimes"]:
        if r["agentRuntimeName"] == runtime_name(task):
            return r["status"]

def state():
    table([(t, "present" if image_exists(t) else "missing", runtime_status(t) or "missing") for t in TASKS], ["task", "image", "runtime"])

state()

In [ ]:
def build_phase():
    try:
        ids = cb.list_builds_for_project(projectName=PREFIX + "build", sortOrder="DESCENDING")["ids"][:len(TASKS)]
        return ", phases: " + ", ".join(b["currentPhase"] for b in cb.batch_get_builds(ids=ids)["builds"] if b["buildStatus"] == "IN_PROGRESS")
    except Exception:
        return ""

todo = [t for t in TASKS if REBUILD or not image_exists(t)]
if todo:
    print("building", todo, "(2 to 8 minutes)")
    rc, out = p2("build", *todo, until=r"^\[$", tick=build_phase)
else:
    print("All images exist. Set REBUILD = True to build them again.")

In [ ]:
todo = [t for t in TASKS if runtime_status(t) != "READY" or REBUILD]
if todo:
    rc, out = p2("deploy", *todo, until=r"^\[$")
state()

### Check the git seal and the hidden tests in a session

The build file says what it does. This check shows what the agent actually finds. It starts one session of the reject task and one of the coding task, and runs these commands as user `agent`, as the agent would:

- `git log --all` must show one commit, and `git remote` no remote. The agent cannot find the fix in the history.
- `git status` must show the change under review for the PR review task, and no change for the coding task.
- The hidden tests in `/opt/hidden` must not be readable, and `/tests` must not exist yet.

The cell takes about 1 minute, because the two sessions start one after the other. It makes no model calls.

In [ ]:
SEAL = ('cd /work/repo; echo "commits=$(git log --all --oneline | wc -l) remotes=$(git remote | wc -l) '
        'changed_files=$(git status --porcelain | wc -l)"; ls /opt/hidden >/dev/null 2>&1 && echo hidden_tests=READABLE '
        '|| echo hidden_tests=denied; [ -e /tests ] && echo tests_dir=PRESENT || echo tests_dir=absent')
DRIVER_SEAL = '''
import json, sys, p2_v2 as p2
out = {}
for task in sys.argv[1:]:
    s = p2.Session(task)
    s.start("cat /run/fw.status")
    out[task] = " ".join(s.agent(SEAL, 120)[0].split())
    s.stop()
print("RESULT " + json.dumps(out))
'''.replace("SEAL", repr(SEAL))
proc = subprocess.run([sys.executable, "-c", DRIVER_SEAL, "pr4615", "pr371"], cwd=HERE, env=ENV,
                      capture_output=True, text=True, timeout=600)
line = next((l for l in proc.stdout.splitlines() if l.startswith("RESULT ")), None)
if line is None:
    raise RuntimeError("The seal check failed: " + mask(proc.stderr[-600:] or proc.stdout[-600:]))
for task, seen in json.loads(line[7:]).items():
    print(f"{task}: {seen}")

**Read the output.** Both tasks show `commits=1 remotes=0`. The reject task shows the changed files of the PR (the agent reviews them), and the coding task shows `changed_files=0` (the agent starts from the base). Both show `hidden_tests=denied tests_dir=absent`. If one of these is different, stop: no score of that task can be trusted.

## 6. Run One Trial of Each Kind

`run TASK:N` starts N trials of a task, each in its own session. Each trial does these steps, and the runner times each one:

| Step | What happens |
|---|---|
| `session_start` | AgentCore starts the microVM. A read-only probe checks that the firewall is up |
| `agent_prepare` | The runner sets the allowlist to the agent's model endpoint, gets the agent role credentials, and writes the instruction |
| `agent_loop` | Codex works as user `agent` inside the firewall |
| `phase_switch` | Agent processes are killed, the allowlist and the credentials change, the hidden tests are copied in |
| `verifier` | `tests/test.sh` runs the grader |
| `readback` | The logs and the results are copied to `jobs/<job>/<trial>/` on your computer |

Each run writes to a new folder `jobs/nb-<time>-<name>/`, so you can run a cell again and keep the earlier results.

The first cell below defines what the notebook shows for a trial. It does not call AWS.

In [ ]:
STEPS = {"session_start": "microVM starts, firewall probe", "agent_prepare": "allowlist, credentials, instruction",
         "agent_loop": "the agent works", "phase_switch": "kill agent, switch allowlist and role, copy tests",
         "verifier": "the grader runs", "readback": "copy results to your computer", "total": "the whole trial"}

def new_job(name):
    return f"jobs/nb-{time.strftime('%H%M%S')}-{name}"

def trial_dirs(job):
    return sorted(p.parent for p in Path(job).glob("*/result.json"))

def show_timings(trial):
    res = read_json(trial / "result.json")
    table([(k, res["timings"].get(k), STEPS[k]) for k in STEPS if k in res["timings"]], ["step", "seconds", "what"])
    return res

def review_row(trial):
    res = read_json(trial / "result.json")
    expected = read_toml(TASK_DIRS[res["task"]] / "task.toml")["metadata"]["harvest"]["expected_verdict"]
    path = trial / "artifacts" / "findings.json"
    findings = read_json(path) if path.exists() else []
    predicted = "block" if any(f.get("severity") == "high" for f in findings) else "approve"
    reward = res.get("reward")
    ok = bool(reward) and predicted == expected and (expected == "approve" or reward.get("coverage_required") == 1.0)
    return {"task": res["task"], "expected": expected, "predicted": predicted, "pass": ok, "reward": reward,
            "findings": findings, "errors": res["errors"]}

def show_review(trial):
    r = review_row(trial)
    status = "NO REWARD (invalid trial)" if r["reward"] is None else ("PASS" if r["pass"] else "FAIL")
    print(f"{status}: {r['task']}, expected {r['expected']}, the agent predicted {r['predicted']}")
    for k in ("coverage_required", "coverage_all", "uncredited"):
        print(f"   {k} = {(r['reward'] or {}).get(k)}")
    for f in r["findings"]:
        print(f"   [{f.get('severity')}] {str(f.get('path', '')).split('/')[-1]}:{f.get('line')} {str(f.get('statement', ''))[:100]}")
    for e in r["errors"]:
        print("   error:", e[:200])

def show_fix(trial):
    details = read_json(trial / "verifier" / "reward_details.json")
    flat = details.get("flat", {})
    status = "INVALID" if details.get("invalid") else ("TASTEFUL SOLVE" if flat.get("reward") == 1.0 else
                                                      "CORRECT, NOT TASTEFUL" if flat.get("correct") == 1.0 else "FAIL")
    print(f"{status}: {read_json(trial / 'result.json')['task']}")
    for gate, ok in details.get("checks", {}).items():
        print(f"   {'pass' if ok else 'FAIL'}  {gate}")
    print("   " + ", ".join(f"{k}={flat.get(k)}" for k in ("fail_to_pass", "pass_to_pass", "validation_score", "rubric_score",
                                                           "relative_taste", "practice_alignment", "patch_bloat")))
    for line in details.get("failed_tests", [])[:5]:
        print("   failed test:", line)
    for name, check in (details.get("taste", {}).get("dimensions") or {}).items():
        if isinstance(check, dict) and check.get("passed") is False:
            print(f"   taste check failed: {name}: {str(check.get('reason', ''))[:140]}")

def show_transcript(trial, n=12):
    commands, last = [], ""
    for line in (trial / "agent" / "codex.txt").read_text(errors="ignore").splitlines():
        try:
            event = json.loads(line)
        except ValueError:
            continue
        item = event.get("item") or {}
        if event.get("type") == "item.started" and item.get("type") == "command_execution":
            commands.append(" ".join(item["command"].split())[:130])
        if event.get("type") == "item.completed" and item.get("type") == "agent_message":
            last = item.get("text", "")
    print(f"the agent ran {len(commands)} commands. The first {min(n, len(commands))}:")
    for c in commands[:n]:
        print("   $", mask(c))
    print("\nits last message:", mask(last[:600].replace("\n", " ")))

def verifier_timeline(trial):
    # the modification time of each result file shows when its stage ended (seconds after the first file)
    rows = [l.split(" ", 2) for l in read_json(trial / "result.json").get("verifier_files_mtime", "").splitlines() if l.strip()]
    keep = ("agent.patch", "junit.xml", "agent_tests.json", "validation.json", "rubric.json", "taste.json", "bloat.json", "reward.json")
    rows = sorted((float(t), p.rsplit("/", 1)[-1]) for t, _, p in rows if p.rsplit("/", 1)[-1] in keep)
    table([(f"+{t - rows[0][0]:.0f}", name) for t, name in rows], ["seconds", "file written"]) if rows else None

### One PR review trial (about 1 minute, about $0.13)

The reject task (PR 4615) is the one from section 4. Codex reads the change, runs the tests, tries inputs, and writes `findings.json`.

In [ ]:
job_review = new_job("review")
rc, out = p2("run", "pr4615:1", "--job", job_review)

In [ ]:
trial = trial_dirs(job_review)[0]
show_review(trial)
res = show_timings(trial)
print("\nfirewall probe at the start of the session:", res["start_probe"])

**Read the result.** A pass needs the verdict `block` and `coverage_required` = 1.0. Codex passes this task in only about 12 % of trials (23 of 185 in the measured sample). That is the agent and the judge, not a fault of the environment, so a fail here is normal. Do not report a low pass rate on this task as a bug. Section 10 says how many trials you need before a rate means something.

The next cell shows how to read what the agent did. The folder of a trial has `result.json` (the runner's record: timings, errors, the reward), `agent/` (what the agent did), `artifacts/` (the findings of a review), and `verifier/` (every file that the grader wrote).

In [ ]:
print("files of the trial:")
for p in sorted(trial.rglob("*")):
    if p.is_file() and "codex-home" not in p.parts:
        print("  ", p.relative_to(trial))
print()
show_transcript(trial)

### One coding trial (3 to 7 minutes, about $0.40)

The agent writes a fix, then the six stages of the grader run. The agent takes 2 to 4 minutes and the grader 1.5 to 2.5 minutes. Look at the timings: the microVM starts in about 1 second and the firewall adds no wait.

In [ ]:
job_coding = new_job("coding")
rc, out = p2("run", "pr371:1", "--job", job_coding)

In [ ]:
trial = trial_dirs(job_coding)[0]
show_fix(trial)
show_timings(trial)
verifier_timeline(trial)

### How to read a reward.json

`verifier/reward.json` holds only numbers, one key for each signal. `verifier/reward_details.json` adds why: `checks` (each gate of the tasteful solve, pass or fail), `flat` (the same numbers as `reward.json`), `failed_tests`, the validation stories, each taste check with its reason, each rubric criterion, and the models that ran. Read it in this order:

1. Is there a `reward.json` at all? If not, the trial is invalid: read `invalid` in `reward_details.json`.
2. `correct`: 0 means the fix does not work. Read `failed_tests` and `pytest.log`.
3. `reward`: 0 with `correct` = 1 means the fix works but a taste gate failed. Read `checks`, then `taste.json` and `rubric.json`.

In [ ]:
print("--- verifier/reward.json ---")
print(json.dumps(read_json(trial / "verifier" / "reward.json"), indent=1))
details = read_json(trial / "verifier" / "reward_details.json")
print("\nreward_details.json keys:", sorted(details))
print("checks:", details.get("checks"))

Each gate of the tasteful solve is listed above. A reward of 0 does not tell you which gate failed: read the gates. If the trial is **invalid** (no reward), the judge or the grader had a fault, not the agent. The runner says why and gives the command to run it again.

### Open the patch and the transcript

`verifier/agent.patch` is what the agent changed. Compare it with the merged fix. `agent/codex.txt` has one JSON event for each step of the agent. The cell reads it and lists the commands, and `agent/codex-home/sessions/` has the full event log of Codex.

In [ ]:
print("--- verifier/agent.patch (the agent's change) ---")
print((trial / "verifier" / "agent.patch").read_text()[:2200])
print("--- what the agent did (from agent/codex.txt) ---")
show_transcript(trial)

### Examine each failure

The score tells you that the agent failed. The trajectory and the patch tell you why. **Use a coding agent to read the trajectories**: a trajectory has many commands, and a coding agent can read it faster than you can. For example:

```text
Read the trial folders in agentcore/jobs/. For each trial with reward 0, compare verifier/agent.patch
with the task's solution/oracle.patch, read verifier/pytest.log, and read agent/codex.txt. Tell me what
the agent thought the cause was, which tests it ran, and why its fix fails. For a trial that is correct
but not tasteful, read verifier/taste.json and verifier/rubric.json and tell me if you agree with the
failed checks. For a PR review that missed an oracle finding, tell me what the agent read and which
inputs it tried. Do not change files.
```

Each failure has one of these causes:

| Cause | Example | What to do |
|-------|---------|------------|
| **The agent misses what it can find** | The agent read the guard, but it did not try the input that breaks it. Or it changed the evaluator prompt, not the extractor | Use the eval to improve the agent: its prompt, its tools, or its procedure (for example, "write a failing test first") |
| **The agent lacks the reviewer's context** | The reviewer knew how a different service uses the code. The repository does not show this | Give the agent more context: design documents, team rules, the services that use the code |
| **The fix breaks something** | `existing_suite` is less than 1 | The agent did not run the full test suite. Improve its procedure |
| **The fix works, but it is not tasteful** | `correct` is 1, but `relative_taste` is 0.4: the agent patched the symptom, not the cause | Read `taste.json` first. If you do not agree with the judge, change the judge prompt, and run the oracle again |
| **The datapoint is wrong** | The agent blocked an approve task, and a later commit fixed the same defect. Or a hidden test checks one design, not the behavior | Correct the oracle or the test, or select a different PR. Then run the controls again |
| **The grader did not give credit** | A correct finding is one line outside the span of the oracle finding | Make the span wider in `oracle.json` |
| **The environment is broken** | A command failed because a tool is missing | Repair the build file, then build the image again |

## 7. The Controls: Test the Grader Before You Trust the Agent Score

Before you decide that an agent is good or bad, make sure that the grader gives the right score to two known answers:

- **An empty agent** that changes nothing. On a coding task, it must score 0: if an f2p test passes without a fix, it does not test the bug. On a reject task, it must fail: it finds nothing. **On an approve task, it passes**, because "no findings" is the right answer there. This is why the dataset has both kinds: an agent that approves everything passes the approve task, and the reject task catches it.
- **The merged fix** (the oracle) as the agent. On a coding task, it must be a tasteful solve. If it is not, a mechanism of the reward stack is broken. On a review task, `solution/solve.sh` submits the oracle findings, and the score must be perfect.

Both checks use the real grader and cost only the judge calls (about $0.04 for each coding trial). The empty coding trial takes about 1.5 minutes, because the grader still runs its stages. For a review task there is no patch to apply, so this runner has no oracle run for it.

When the tasks were built, both controls were run on both coding tasks:

| Task | Run | f2p | p2p | existing | validation | rubric | taste checks (rel. / practice) | bloat | reward |
|---|---|---|---|---|---|---|---|---|---|
| root agent (371) | oracle | 1.0 | 1.0 | 1.0 | 2/2 stories | 1.0 | 5/5 / 5/5 | 1.0 | **1** |
| root agent (371) | empty | 0.0 | 1.0 | 1.0 | 0/2 | 0.5 | - | - | 0 |
| tool history (338) | oracle | 1.0 | 1.0 | 1.0 | 2/2 stories | 1.0 | 5/5 / 5/5 | 1.0 | **1** |
| tool history (338) | empty | 0.0 | 1.0 | 1.0 | 0/2 | 0.5 | - | - | 0 |

If a p2p test fails for the empty agent, the test or the environment is wrong. Senior SWE-Bench runs each control 3 times and rejects a task unless all 3 results are correct (pass^3).

In [ ]:
job_empty = new_job("empty")
rc, out = p2("run", "pr4615:1", "pr4121:1", "pr371:1", "--agent", "empty", "--job", job_empty)

In [ ]:
EXPECT = {"pr4615": "FAIL", "pr4121": "PASS", "pr371": "FAIL"}
rows = []
for t in trial_dirs(job_empty):
    res = read_json(t / "result.json")
    if res["task"] == "pr371":
        got = "NO REWARD" if res["reward"] is None else ("PASS" if res["reward"].get("reward") == 1.0 else "FAIL")
        detail = f"correct={(res['reward'] or {}).get('correct')}, fail_to_pass={(res['reward'] or {}).get('fail_to_pass')}"
    else:
        r = review_row(t)
        got = "NO REWARD" if r["reward"] is None else ("PASS" if r["pass"] else "FAIL")
        detail = f"findings={len(r['findings'])}, predicted {r['predicted']}"
    rows.append((res["task"], "empty agent", EXPECT[res["task"]], got, "ok" if got == EXPECT[res["task"]] else "UNEXPECTED", detail))
table(sorted(rows), ["task", "agent", "expected", "got", "check", "detail"])

In [ ]:
oracle_patch = CODING_TASK / "solution" / "oracle.patch"
job_oracle = new_job("oracle")
rc, out = p2("run", "pr371:1", "--agent", f"patch:{oracle_patch}", "--job", job_oracle)
trial = trial_dirs(job_oracle)[0]
show_fix(trial)

**Read the result.** The merged fix should be a tasteful solve. The taste judge is an LLM and it varies (section 10), so a single `CORRECT, NOT TASTEFUL` on the oracle is possible but rare: run the cell again before you decide that something is broken. If the oracle fails on the hidden tests, the environment or the tests are wrong, and no agent score can be trusted.

## 8. The Firewall: What the Agent Can and Cannot Reach

The agent runs untrusted code, and it must not find the answer. A normal firewall (nftables, iptables) does not load in the microVM, because root has no network-admin capability there. The runner builds the seal from these parts (`image/start.sh` and `image/proxy.py` are all of it):

1. The agent runs in an **empty network namespace**. It has only a loopback interface, so it has no route to anything.
2. A **relay** on `127.0.0.1:3128` inside that namespace forwards to a unix socket.
3. An **allowlist proxy** on the other side of the socket accepts `CONNECT host:443` for the hosts in a root-owned file only. The runner rewrites that file at each phase: the agent's model endpoint in the agent phase, the judge endpoint in the grader phase.
4. The credentials of each phase can call only the model of that phase.

Two checks show this, both from inside a real session. Neither one tries to break out: for that, you need an expert review. The authors ran 43 scripted probes and two LLM red-team runs and found no break-out, but that is not a certified boundary. The environment is `PUBLIC` network mode, so outside the firewall the microVM has internet access.

### Check 1: what each role can call

`scripts/scope_test_v2.py` calls three models with the credentials of each phase. The grader role must reach the judge model only. The agent role must not reach `bedrock-runtime` at all. It uses one-token calls. It needs the `pr4121` runtime, so run it after section 5.

In [ ]:
rc, out = stream([sys.executable, "-u", "scripts/scope_test_v2.py", "--prefix", PREFIX])

**Read the output.** `OK ok` means that the model answered. `AccessDeniedException` means that the role may not call that model. The grader role reaches the judge model (Haiku) only, and the agent role reaches no model on `bedrock-runtime`: the agent calls its own model through a different service (`bedrock-mantle`) with its own permission.

### Check 2: what the agent can reach, host by host

This probe runs as user `agent` in the agent phase. For each host it makes an HTTPS request through the proxy, as Codex does, and a direct request that skips the proxy. A host that answers with any HTTP status (even `403` from the service for a call without credentials) was **reached**. A `Tunnel connection failed: 403` is the **proxy** saying no. The probe is a few lines of Python that run inside the session.

In [ ]:
PROBE = '''
import json, urllib.request, urllib.error
HOSTS = ["bedrock-mantle.us-east-1.api.aws", "bedrock-runtime.us-east-1.amazonaws.com", "github.com", "pypi.org", "example.com"]
def get(host, proxy):
    opener = urllib.request.build_opener() if proxy else urllib.request.build_opener(urllib.request.ProxyHandler({}))
    try:
        r = opener.open(f"https://{host}/", timeout=8)
        return f"reached, HTTP {r.status}"
    except urllib.error.HTTPError as e:
        return f"reached, HTTP {e.code}"
    except Exception as e:
        text = str(getattr(e, "reason", e))
        return "blocked by the proxy" if "403" in text else "no route (" + text[:40] + ")"
print(json.dumps({h: [get(h, True), get(h, False)] for h in HOSTS}))
'''
DRIVER = '''
import json, sys, p2_v2 as p2
s = p2.Session("pr4121")
s.start("cat /run/fw.status")
s.put("/tmp/probe.py", sys.stdin.read().encode(), "0644", "agent")
out = {}
for phase, host in (("agent", p2.HOST_MANTLE), ("grader", p2.HOST_RUNTIME)):
    p2.set_phase(s, host, {})
    out[phase] = s.agent("python3 /tmp/probe.py", 120)[0].strip().splitlines()[-1]
s.stop()
print("RESULT " + json.dumps(out))
'''
proc = subprocess.run([sys.executable, "-c", DRIVER], input=PROBE, cwd=HERE, env=ENV, capture_output=True, text=True, timeout=600)
line = next((l for l in proc.stdout.splitlines() if l.startswith("RESULT ")), None)
if line is None:
    raise RuntimeError("The probe failed: " + mask(proc.stderr[-600:] or proc.stdout[-600:]))
rows = []
for phase, answers in json.loads(line[7:]).items():
    for host, (via_proxy, direct) in json.loads(answers).items():
        rows.append((phase, host.replace("us-east-1.", "<region>."), via_proxy, direct))
table(rows, ["phase", "host", "through the proxy", "direct, skipping the proxy"])

**Read the table.** In the agent phase, only the model endpoint of the agent is reached. GitHub and PyPI are blocked, so the agent cannot look up the fix and cannot install a package. Skipping the proxy gives no route, because the namespace has only loopback. In the grader phase, the allowlist holds the judge endpoint instead, and the agent's endpoint is closed. The runner reads the grader credentials as root, outside the agent's namespace, and hands them only to the grader. The agent never gets them.

## 9. Parallelism: Many Trials at the Same Time

`run` starts all of its trials at once, each in its own microVM. A batch takes about as long as its slowest trial. In the measured sample, 12 or 24 trials at once took 10 % to 20 % longer than one trial, and 30 at once ran without throttling. Thirty is the largest size that was tested: for more, run waves one after the other (see [TESTER-GUIDE.md](TESTER-GUIDE.md), section 3).

This cell runs 12 PR review trials, 6 of each task (about 1 minute and about $1.60). The cell after it compares the wall clock of the batch with the time that the same trials would take one after the other.

In [ ]:
N_EACH = 6        # trials of each review task. 12 in total. The tested limit is 30 at once
job_batch = new_job("batch")
rc, batch_out = p2("run", f"pr4121:{N_EACH}", f"pr4615:{N_EACH}", "--job", job_batch)

In [ ]:
results = [read_json(t / "result.json") for t in trial_dirs(job_batch)]
wall = float(next(re.search(r"batch wall clock (\d+) s", l).group(1) for l in batch_out if "batch wall clock" in l))
serial = sum(r["timings"]["total"] for r in results)
totals = sorted(r["timings"]["total"] for r in results)
print(f"{len(results)} trials, batch wall clock {wall:.0f} s")
print(f"the same trials one after the other: about {serial:.0f} s ({serial / wall:.1f} times as long)")
print(f"trial time: shortest {totals[0]:.0f} s, median {totals[len(totals) // 2]:.0f} s, longest {totals[-1]:.0f} s")
print(f"throttled calls: {sum(r['throttles'] for r in results)}, sessions that needed a start retry: {sum('start_attempts' in r for r in results)}")
print(f"trials without a reward: {sum(r['reward'] is None for r in results)}")

The wall clock is longer than the longest trial by the time for the batch to start and for the slowest result to come back. The runner retries the first call of a session up to 3 times on a new session when AgentCore fails to start it (a log line says so). Later calls are never retried, so the agent's work is never repeated. A trial that still has no reward is listed under the table with the reason and the command to run it again.

### The dataset metrics

The batch has 6 trials of each review task. The cell applies the pass rule of section 4.4 to each trial, and calculates the metrics of the dataset. With so few trials, read them as an example of the calculation, not as a measure of the agent (section 10).

In [ ]:
rows = [review_row(t) for t in trial_dirs(job_batch)]
rows = [r for r in rows if r["reward"] is not None]
must_block = [r for r in rows if r["expected"] == "block"]
good = [r for r in rows if r["expected"] == "approve"]
rate = lambda part, whole: f"{len(part)} of {len(whole)}" if whole else "n/a"
table([("accuracy", rate([r for r in rows if r["pass"]], rows)),
       ("block recall", rate([r for r in must_block if r["predicted"] == "block"], must_block)),
       ("false approve rate", rate([r for r in must_block if r["predicted"] == "approve"], must_block)),
       ("false block rate", rate([r for r in good if r["predicted"] == "block"], good))], ["metric", "trials"])

## 10. Read the Results With the Noise in Mind

An eval score is a measurement with an error. **The results change from run to run**: the agent can find a defect in one run and miss it in the next, and the LLM judges can give a different answer to the same question. A failure that occurs in each trial is a weakness. A failure that occurs only sometimes is variance. These are the sources of noise, and their size in samples measured with this runner:

| Source | Size | What to do |
|---|---|---|
| **Few trials** | The reject task passes about 12 % of the time with Codex (23 of 185 trials, 95 % interval 8 % to 18 %). With 6 trials, the 95 % interval around an observed rate is very wide | Run 20 or more trials before you call a rate different from 12 %. About 85 trials separate 12 % from 30 % |
| **The taste judge** | For one patch, the score varied from 0.4 to 1.0 (27 repeats, mean 0.80, standard deviation about 0.19) | Do not read a single taste score. A score of 0.4 is under the gate, so a correct fix can get reward 0 from this judge alone. Look at the gates over many trials |
| **Lost trials** | 7 of 60 trials (about 12 %) had no reward in a run of 60 at once: 4 failed session starts (the runner now retries them), 2 lost in the read back (now fixed), 1 judge fault | They are not agent fails. Run them again |

The cell below pools all review trials that you ran in this notebook and shows the pass rate with a 95 % Wilson interval. With 15 or so trials, the interval is wide, and it should be.

In [ ]:
from math import sqrt
from collections import defaultdict

def wilson(k, n, z=1.96):
    if n == 0:
        return (0.0, 1.0)
    p, d = k / n, 1 + z * z / n
    mid, half = (p + z * z / (2 * n)) / d, z * sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return max(0.0, mid - half), min(1.0, mid + half)

per_task = defaultdict(lambda: [0, 0, 0])      # trials with a reward, passes, trials without a reward
for job in sorted(Path("jobs").glob("nb-*")):
    for t in trial_dirs(job):
        if "empty" in job.name or "oracle" in job.name or read_json(t / "result.json")["task"] not in ("pr4121", "pr4615"):
            continue
        r = review_row(t)
        per_task[r["task"]][2] += r["reward"] is None
        if r["reward"] is not None:
            per_task[r["task"]][0] += 1
            per_task[r["task"]][1] += r["pass"]
rows = []
for task, (n, k, lost) in sorted(per_task.items()):
    lo, hi = wilson(k, n)
    rows.append((task, TASK_DIRS[task].name.split("-")[-1], f"{k} of {n}", f"{100 * lo:.0f} % to {100 * hi:.0f} %", lost))
table(rows, ["task", "kind", "passed", "95 % interval", "no reward"])

**What this tells you.** The approve task passes in nearly every trial (21 of 21 in the first measured sample), and the reject task passes rarely. Together they show that the agent does not approve everything, and does not block everything. A pass rate of the reject task far from 12 % over 20 or more trials is worth a report. Anything smaller is inside the noise.

**What it does not tell you.** The measured samples are from one account, one day, and one agent. They do not show how a different model, or a different task, would do. In the coding sample (60 trials), 50 of 53 graded trials were a tasteful solve. A reward that is almost always 1 cannot show a worse agent: it needs harder tasks.

## 11. Clean Up

> **Warning: run this section when you are done.** It deletes every resource with the prefix `evals-nb-` (or your `PREFIX`) and the tag `Project=evals-streamline-spike` that this notebook created: runtimes, images, the CodeBuild project, the three roles, the S3 bucket, and the log groups. It never touches a name that does not start with your prefix. The cells in this section do not need the earlier cells: you can run them in a fresh kernel.

A runtime takes 3 to 9 minutes to delete. The cell waits for it, then checks every list. If your policy blocks a delete, the cell prints the commands for you to run by hand.

In [ ]:
import json, os, re, subprocess, sys, time
from pathlib import Path
import boto3

PREFIX = os.environ.get("P2_PREFIX", "evals-nb-")        # the same prefix as in section 2
DELETE_LOCAL_JOBS = False                                # True also deletes the trial folders jobs/nb-* on your computer
REGION = "us-east-1"
assert PREFIX.startswith("evals-"), "refusing a prefix that does not start with evals-"
HERE = Path.cwd()
mask = lambda s: re.sub(r"(?<!\d)\d{12}(?!\d)", "ACCT", s).replace(str(Path.home()), "~")
sess = boto3.Session()
cl = lambda n: sess.client(n, region_name=REGION)
UP = re.sub(r"[^A-Za-z0-9_]", "_", PREFIX)               # runtime names use underscores
account = sess.client("sts").get_caller_identity()["Account"]
BUCKET = f"{PREFIX}src-{account}-{REGION}"

In [ ]:
# Step 1: the runner deletes the runtimes, the ECR repository and images, the CodeBuild project, and the three roles
proc = subprocess.run([sys.executable, "-u", "p2_v2.py", "--prefix", PREFIX, "cleanup"], cwd=HERE, capture_output=True, text=True,
                      env={**os.environ, "P2_PREFIX": PREFIX})
print(mask(proc.stdout + proc.stderr).strip())
print("cleanup exit code:", proc.returncode, "(1 means that something is still busy: the next cells wait and run it again)")

In [ ]:
# Step 2: wait for the runtimes to be gone (a delete takes 3 to 9 minutes), and run the runner again if one was busy
def runtimes():
    return [r for r in cl("bedrock-agentcore-control").list_agent_runtimes(maxResults=100)["agentRuntimes"]
            if r["agentRuntimeName"].startswith(UP)]

t0, reran = time.time(), False
while runtimes() and time.time() - t0 < 900:
    now = runtimes()
    print(f"   {time.time() - t0:.0f} s: " + ", ".join(f"{r['agentRuntimeName']} {r['status']}" for r in now), flush=True)
    if proc.returncode and not reran and not any(r["status"] in ("CREATING", "UPDATING") for r in now):
        reran = True                                      # the runtimes that were busy can be deleted now
        proc = subprocess.run([sys.executable, "-u", "p2_v2.py", "--prefix", PREFIX, "cleanup"], cwd=HERE, capture_output=True,
                              text=True, env={**os.environ, "P2_PREFIX": PREFIX})
        print(mask(proc.stdout + proc.stderr).strip())
    time.sleep(30)
print("runtimes left:", [r["agentRuntimeName"] for r in runtimes()] or "none")

In [ ]:
# Step 3: what the runner does not delete: the S3 bucket and the log groups
by_hand = []
def attempt(what, command, fn):
    try:
        fn(); print("deleted", what)
    except Exception as e:
        code = getattr(e, "response", {}).get("Error", {}).get("Code", type(e).__name__)
        if code in ("NoSuchBucket", "ResourceNotFoundException"):
            return
        print(f"could not delete {what}: {code}"); by_hand.append(command)

s3 = sess.resource("s3", region_name=REGION)
if BUCKET in [b["Name"] for b in cl("s3").list_buckets()["Buckets"]]:
    attempt(f"bucket {mask(BUCKET)}", f"aws s3 rm s3://{BUCKET} --recursive && aws s3api delete-bucket --bucket {BUCKET} --region {REGION}",
            lambda: (s3.Bucket(BUCKET).objects.all().delete(), cl("s3").delete_bucket(Bucket=BUCKET)))
logs = cl("logs")
for pre in (f"/aws/codebuild/{PREFIX}", f"/aws/bedrock-agentcore/runtimes/{UP}"):
    for g in logs.describe_log_groups(logGroupNamePrefix=pre)["logGroups"]:
        attempt(f"log group {g['logGroupName']}", f"aws logs delete-log-group --log-group-name '{g['logGroupName']}' --region {REGION}",
                lambda g=g: logs.delete_log_group(logGroupName=g["logGroupName"]))

### Check that nothing is left

Every list must be empty. The last one is a query by tag: it lists every resource with the tag `Project=evals-streamline-spike`, so it also shows the resources of other prefixes (yours or a colleague's). Look for your prefix only. A runtime that was just deleted can show for a minute in the tag query. Run the cell again if it does.

In [ ]:
iam = cl("iam")
left = {
    "runtimes": [r["agentRuntimeName"] for r in runtimes()],
    "iam roles": [r["RoleName"] for pg in iam.get_paginator("list_roles").paginate() for r in pg["Roles"] if r["RoleName"].startswith(PREFIX)],
    "ecr repositories": [r["repositoryName"] for pg in cl("ecr").get_paginator("describe_repositories").paginate()
                         for r in pg["repositories"] if r["repositoryName"].startswith(PREFIX)],
    "codebuild projects": [p for p in cl("codebuild").list_projects()["projects"] if p.startswith(PREFIX)],
    "s3 buckets": [b["Name"] for b in cl("s3").list_buckets()["Buckets"] if b["Name"].startswith(PREFIX)],
    "log groups": [g["logGroupName"] for pre in (f"/aws/codebuild/{PREFIX}", f"/aws/bedrock-agentcore/runtimes/{UP}")
                   for g in logs.describe_log_groups(logGroupNamePrefix=pre)["logGroups"]],
    "tagged resources of this prefix": [a for pg in cl("resourcegroupstaggingapi").get_paginator("get_resources").paginate(
        TagFilters=[{"Key": "Project", "Values": ["evals-streamline-spike"]}]) for m in pg["ResourceTagMappingList"]
        for a in [m["ResourceARN"]] if PREFIX in a or UP in a],
}
for name, items in left.items():
    print(f"{'OK ' if not items else 'LEFT'} {name}: {[mask(i) for i in items] or 'none'}")
if DELETE_LOCAL_JOBS:
    import shutil
    [shutil.rmtree(p) for p in HERE.glob("jobs/nb-*")]
print("\nEverything is deleted." if not any(left.values()) else "\nSomething is left: read the next cell.")

### If something is left: delete it by hand

Do this only if the cell above shows `LEFT`. Most often, a runtime is still in `DELETING` (wait, and run the check cell again), or your policy does not allow `s3:DeleteObject`, `s3:DeleteBucket`, or `logs:DeleteLogGroup`. The cell prints the commands for the S3 bucket and the log groups. It never prints a command that deletes anything that does not start with your prefix. The AWS CLI is needed only for these commands. Replace `ACCT` with your account ID.

If a role, the repository, or the project is left, `python3 p2_v2.py --prefix <your prefix> cleanup` in a terminal prints the reason, and `scripts/teardown.sh <your prefix>` (or with `--print`) does the same checks with the AWS CLI.

In [ ]:
if by_hand:
    print("Run these in a terminal with a role that may delete them:\n")
    print("\n".join(mask(c) for c in by_hand))
else:
    print("No commands to run by hand." if not any(left.values()) else
          "Nothing to print. Wait for the runtimes, then run the check cell again, or use scripts/teardown.sh.")

## Summary

- Your git history contains your eval data. Roll back to the parent commit, seal the environment, and use the PR review or the merged patch as the oracle.
- The task format keeps the agent files (`instruction.md`, `environment/`) apart from the grader files (`tests/`, `solution/`). The tests are hidden, so the agent cannot write code that passes them and nothing else.
- A PR review passes on the right verdict and full required coverage. A reject datapoint and an approve datapoint together find an agent that blocks or approves everything.
- A fix is a tasteful solve only if it is correct (hidden tests, existing suite, validation stories) and passes the rubric, taste, and bloat gates. Read the gates, not only the number.
- The judges are a different model from the agent, and each phase has credentials for its own model only.
- In AgentCore, each trial is one microVM. The seal is the task's build file (git), a firewall inside the session (network), and a root-only copy of the tests (grader). Many trials run at the same time.
- Run the controls first: an empty agent must fail a coding task and a reject task, and the merged fix must be a tasteful solve.
- A rate from a few trials is noise. A trial without a reward is a grader or an infrastructure fault, not an agent fail.
- Always run the cleanup, and check the lists.

**More:** [README.md](README.md) (design and commands) and [TESTER-GUIDE.md](TESTER-GUIDE.md) (permissions, cost, limits, feedback template) in this folder.